# KronLLM Q10A — decay pass (WSD tail)

Resumes the end-of-epoch Q10 weights and anneals the LR 3e-4 → 1e-6 over
**29.6 % of an epoch** (40,649 steps × 16,384 tokens = 666M tokens) on the
curated mix: literature, encyclopedic, science, 3× more math than the epoch saw,
and the whole identity slice repeated to ~2.5 % of the tokens. The step count is
the bin divided by the step size, so the pass reads its data exactly once.

No warmup: the pass starts at full LR on the curve. Checkpoints every 250 steps
to `/kaggle/working/q10a`; the session stops at the 500-minute wall clock and
the next one resumes from `last.pt`.


In [ ]:
import glob
import os
import shutil
import subprocess
import sys

print(sorted(os.listdir('/kaggle/input')))
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
for line in open('/proc/meminfo'):
    if line.startswith(('MemTotal', 'MemAvailable')):
        print(line.strip())
try:
    print('cgroup max', open('/sys/fs/cgroup/memory.max').read().strip())
except OSError:
    pass
print('working disk', shutil.disk_usage('/kaggle/working').free // 2**30, 'GiB free')

# Dataset mounts moved to /kaggle/input/datasets/<owner>/<slug> on newer
# images; support both layouts.
def find_dataset(slug, probe):
    roots = glob.glob(f'/kaggle/input/datasets/*/{slug}') + [f'/kaggle/input/{slug}']
    for root in roots:
        if os.path.exists(os.path.join(root, probe)):
            return root
    raise FileNotFoundError(f'{slug}: {probe} not found under /kaggle/input')

SRC = find_dataset('kron-q10a-anneal-bin', 'train.bpe.bin')
CODE = find_dataset('kron-llm-code-v14', 'kronllm')
print('SRC', SRC)
print('CODE', CODE)
sys.path.insert(0, CODE)

# The bins live on the dataset mount; a disk-backed copy keeps two
# DDP ranks under the 32 GB cgroup (tmpfs pages are charged twice).
# /kaggle/working, which is saved as kernel output and would upload 4 GB).
# through that mount were OOM-killed; a local copy is cheap and predictable.
DATA = '/kaggle/working/data'
os.makedirs(DATA, exist_ok=True)
for name in ('train.bpe.bin', 'val.bpe.bin', 'tokenizer.json', 'tokenizer_meta.json'):
    src = os.path.join(SRC, name)
    dst = os.path.join(DATA, name)
    if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
        print('copying', name, round(os.path.getsize(src) / 1e9, 2), 'GB')
        shutil.copyfile(src, dst)
print('local bins', {p: round(os.path.getsize(os.path.join(DATA, p)) / 1e9, 2)
                     for p in ('train.bpe.bin', 'val.bpe.bin')})

import kronllm
print('kronllm', kronllm.__file__)

In [ ]:
import json
meta = json.load(open(DATA + '/tokenizer_meta.json'))
print('vocab', meta['vocab_size'], 'chat ids', meta.get('chat_tokens'))
print('train tokens', meta['train_tokens'], 'val tokens', meta['val_tokens'])
print('bytes/token', meta['bytes_per_token'])
print('token share', meta.get('token_share'), 'byte share', meta.get('byte_share'))

In [ ]:
import torch

# DDP is unstable in this image: two ranks each map the 4.3 GB bin and the
# 32 GB cgroup kills one of them at the first collective.  One T4 with fp16
# and batch 32 is the measured configuration (6.5k tokens/s, 15.2 GB VRAM).
n_gpu = 1
print('GPUs available:', torch.cuda.device_count(), '| using', n_gpu)

cmd = [sys.executable, '-u', '-m', 'kronllm.cli', 'train', '--exp', 'Q10A',
       '--data-dir', DATA, '--run-dir', '/kaggle/working/q10a',
       '--override', 'training.max_minutes=500',
       '--override', 'training.log_every=50']

resume = sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True))
if resume:
    cmd += ['--resume', resume[0]]
    print('resuming from', resume[0])
print(' '.join(cmd))
env = dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED='1',
           CUDA_VISIBLE_DEVICES='0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
try:
    subprocess.run(cmd, check=True, env=env)
finally:
    # Drop the 4.3 GB local copy so it is not uploaded as output.
    shutil.rmtree(DATA, ignore_errors=True)


In [ ]:
print(open('/kaggle/working/q10a/results.json').read()[:1200])
for name in ('best.pt', 'last.pt', 'log.csv'):
    p = '/kaggle/working/q10/' + name
    if os.path.exists(p):
        print(name, round(os.path.getsize(p) / 1e6, 1), 'MB')
print('download last.pt and upload it as a dataset to resume next session')